# 03_phase1_QC_V2.ipynb — FIXED
Phase 1 QC — V(D)J + constant region filtering, MT gene removal, per-sample Scrublet, adata.raw

**Fix applied (this session):** removed the dask/zarr wrapping that was causing `MemoryError` on the GSE176078 load. `da.from_array()` was calling `.copy()` internally on the sparse matrix, which is what blew up memory — dask wasn't actually needed here since the chunked processing functions below already work directly on scipy sparse matrices. Data now stays as scipy CSR sparse throughout, which is more memory-efficient anyway.

In [ ]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import gc
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scrublet as scr
import scanpy.external as sce
from scipy.sparse import issparse, csr_matrix, vstack
from pathlib import Path

sc.settings.verbosity = 1

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RAW_DIR = PROJECT_DIR / "Data" / "Raw"
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase1_qc_v2"
RESULTS_DIR = PROJECT_DIR / "results" / "phase1_qc_v2"

for d in [PROCESSED_DIR, FIGURE_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paths set up")

In [ ]:
# ----------------------------
# Cell 2 — Load raw data
# ----------------------------
adata1 = sc.read_h5ad(RAW_DIR / "GSE114725_raw.h5ad")
adata2 = sc.read_h5ad(RAW_DIR / "GSE176078_raw.h5ad")

print("Loaded:")
print(adata1)
print(adata2)

In [ ]:
# ----------------------------
# Cell 3 — Cast to float32 sparse CSR (no dask — see note above)
# Keeping matrices as plain scipy sparse is memory-efficient on its own;
# dask/zarr added overhead without benefit for this pipeline size.
# FIX: force CSR (not CSC) explicitly. GSE176078_raw.h5ad was saved as
# CSC because 02_GSE176078_data_loading.ipynb transposes a CSR matrix
# (adata = adata.T), which scipy always returns as CSC. Every chunked
# function below does row slicing (adata.X[start:end]), which is slow
# on CSC and fast on CSR — this was likely contributing to the earlier
# Scrublet slowness/crashes on GSE176078 specifically.
# ----------------------------
def to_sparse_float32(adata):
    if issparse(adata.X):
        adata.X = adata.X.tocsr().astype("float32", copy=False)
    else:
        adata.X = csr_matrix(adata.X, dtype=np.float32)
    return adata

adata1 = to_sparse_float32(adata1)
gc.collect()
print("adata1 done:", adata1.X)

adata2 = to_sparse_float32(adata2)
gc.collect()
print("adata2 done:", adata2.X)

print("Sparse float32 arrays ready")

In [ ]:
# ----------------------------
# Cell 4 — Filter V(D)J and constant region genes
# and remove mitochondrial genes
# ----------------------------
# V(D)J variable region genes
vdj_prefixes = (
    "IGHV", "IGLV", "IGKV",
    "TRAV", "TRBV", "TRGV", "TRDV",
    "IGHD", "IGHJ", "IGLJ", "IGKJ"
)

# Constant region genes (added per supervisor feedback)
constant_genes = (
    "TRAC", "TRBC", "TRGC", "TRDC",
    "IGHA", "IGHD", "IGHE", "IGHG", "IGHM",
    "IGLC", "IGKC"
)

all_prefixes = vdj_prefixes + constant_genes

def filter_immune_genes(adata, dataset_name):
    before = adata.n_vars

    # Filter V(D)J and constant region genes
    vdj_mask = ~adata.var_names.str.startswith(all_prefixes)

    # Filter mitochondrial genes completely
    mt_mask = ~adata.var_names.str.startswith("MT-")

    combined_mask = vdj_mask & mt_mask
    adata = adata[:, combined_mask].copy()

    removed_vdj = (~vdj_mask).sum()
    removed_mt = (~mt_mask).sum()

    print(f"{dataset_name}:")
    print(f"  V(D)J + constant region genes removed: {removed_vdj}")
    print(f"  Mitochondrial genes removed: {removed_mt}")
    print(f"  Genes remaining: {adata.n_vars}")

    return adata

adata1 = filter_immune_genes(adata1, "GSE114725")
adata2 = filter_immune_genes(adata2, "GSE176078")

gc.collect()

In [ ]:
# ----------------------------
# Cell 5 — QC metrics (chunk by chunk)
# Note: MT% calculated before MT gene removal for filtering purposes
# Since we removed MT genes in Cell 4, we calculate QC metrics on
# genes/counts only (no MT% column downstream)
# ----------------------------
def calculate_qc_chunked(adata, dataset_name, chunk_size=2000):
    print(f"Calculating QC metrics for {dataset_name}...")

    n_cells = adata.n_obs
    n_genes_by_counts = np.zeros(n_cells, dtype=np.float32)
    total_counts = np.zeros(n_cells, dtype=np.float32)

    for start in range(0, n_cells, chunk_size):
        end = min(start + chunk_size, n_cells)

        chunk = adata.X[start:end]
        if not issparse(chunk):
            chunk = csr_matrix(chunk)

        total_counts[start:end] = np.asarray(
            chunk.sum(axis=1)
        ).flatten()
        n_genes_by_counts[start:end] = np.asarray(
            (chunk > 0).sum(axis=1)
        ).flatten()

        if start % 10000 == 0:
            print(f"  Processed {end}/{n_cells} cells...")

    adata.obs["n_genes_by_counts"] = n_genes_by_counts
    adata.obs["total_counts"] = total_counts

    print(f"  Done. Mean genes: {n_genes_by_counts.mean():.0f}")
    print(f"  Mean counts: {total_counts.mean():.0f}")

    gc.collect()
    return adata

adata1 = calculate_qc_chunked(adata1, "GSE114725")
adata2 = calculate_qc_chunked(adata2, "GSE176078")

print("\nGSE114725 summary:")
print(adata1.obs[["n_genes_by_counts", "total_counts"]].describe())
print("\nGSE176078 summary:")
print(adata2.obs[["n_genes_by_counts", "total_counts"]].describe())

In [ ]:
# ----------------------------
# Cell 6 — QC plots BEFORE filtering
# Note: MT% not plotted as MT genes have been removed
# Cell filtering based on gene count and total counts only
# ----------------------------
for adata, name in [(adata1, "GSE114725"), (adata2, "GSE176078")]:
    sc.pl.violin(
        adata,
        ["n_genes_by_counts", "total_counts"],
        jitter=0.4,
        multi_panel=True,
        show=False
    )
    plt.savefig(
        FIGURE_DIR / f"{name}_before_filtering_violin.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    sc.pl.scatter(
        adata,
        x="total_counts",
        y="n_genes_by_counts",
        show=False
    )
    plt.savefig(
        FIGURE_DIR / f"{name}_counts_vs_genes_scatter.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

print("Before filtering plots saved")

In [ ]:
# ----------------------------
# Cell 7 — Filter low quality cells and genes
# Note: MT% threshold not applied as MT genes removed in Cell 4
# Thresholds based on visual inspection of QC distributions
# ----------------------------
print("Before filtering:")
print(f"GSE114725: {adata1.n_obs} cells, {adata1.n_vars} genes")
print(f"GSE176078: {adata2.n_obs} cells, {adata2.n_vars} genes")

# GSE114725 — immune enriched
# mean genes = 611, threshold removes empty droplets
adata1 = adata1[
    adata1.obs["n_genes_by_counts"] > 200
].copy()

# GSE176078 — full TME
# mean genes = 1776, higher threshold appropriate
adata2 = adata2[
    adata2.obs["n_genes_by_counts"] > 500
].copy()

# Filter lowly expressed genes
sc.pp.filter_genes(adata1, min_cells=3)
sc.pp.filter_genes(adata2, min_cells=3)

print("\nAfter filtering:")
print(f"GSE114725: {adata1.n_obs} cells, {adata1.n_vars} genes")
print(f"GSE176078: {adata2.n_obs} cells, {adata2.n_vars} genes")

gc.collect()

In [ ]:
# ----------------------------
# Cell 8 — Per-sample Scrublet doublet detection
# Run per sample as doublets only form within a sample
# FIX: removed unconditional ".compute()" call (leftover from dask) —
# adata.X is now plain scipy sparse, so we index it directly.
# ----------------------------
def run_scrublet_per_sample(adata, dataset_name, sample_key, chunk_size=2000):
    print(f"Running per-sample Scrublet for {dataset_name}")

    samples = adata.obs[sample_key].unique()
    print(f"  Found {len(samples)} samples")

    all_doublet_scores = np.zeros(adata.n_obs, dtype=np.float32)
    all_predicted_doublets = np.zeros(adata.n_obs, dtype=bool)

    for sample in samples:
        sample_mask = adata.obs[sample_key] == sample
        sample_idx = np.where(sample_mask)[0]
        n_cells = len(sample_idx)

        print(f"  Sample {sample}: {n_cells} cells")

        if n_cells < 50:
            print(f"    Skipping — too few cells")
            continue

        chunks = []
        for start in range(0, n_cells, chunk_size):
            end = min(start + chunk_size, n_cells)
            global_idx = sample_idx[start:end]
            chunk = adata.X[global_idx]
            if not issparse(chunk):
                chunk = csr_matrix(chunk)
            chunks.append(chunk)

        X_sample = vstack(chunks)

        try:
            scrub = scr.Scrublet(X_sample)
            doublet_scores, predicted_doublets = scrub.scrub_doublets(
                verbose=False
            )
            all_doublet_scores[sample_idx] = doublet_scores
            all_predicted_doublets[sample_idx] = predicted_doublets
            n_doublets = predicted_doublets.sum()
            print(f"    Doublets: {n_doublets} ({100*n_doublets/n_cells:.1f}%)")

        except Exception as e:
            print(f"    Scrublet failed for {sample}: {e}")
            continue

        gc.collect()

    adata.obs["doublet_score"] = all_doublet_scores
    adata.obs["predicted_doublet"] = all_predicted_doublets

    before = adata.n_obs
    adata = adata[~adata.obs["predicted_doublet"]].copy()

    print(f"\n  Cells before: {before}")
    print(f"  Cells after: {adata.n_obs}")
    print(f"  Doublets removed: {before - adata.n_obs}")

    sc.pl.violin(adata, ["doublet_score"], jitter=0.4, show=False)
    plt.savefig(
        FIGURE_DIR / f"{dataset_name}_scrublet_scores.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()

    gc.collect()
    return adata

adata1 = run_scrublet_per_sample(adata1, "GSE114725", sample_key="patient")
adata2 = run_scrublet_per_sample(adata2, "GSE176078", sample_key="orig.ident")

In [ ]:
# ----------------------------
# Cell 9 — Normalisation
# FIX: removed unconditional ".compute()" calls (leftover from dask)
# ----------------------------
def check_and_normalise(adata, dataset_name, chunk_size=2000):
    # Check larger sample for more reliable detection
    n_check = min(5000, adata.n_obs)
    chunks = []
    for start in range(0, n_check, chunk_size):
        end = min(start + chunk_size, n_check)
        chunk = adata.X[start:end]
        if not issparse(chunk):
            chunk = csr_matrix(chunk)
        chunks.append(chunk)
    X_check = vstack(chunks)

    max_val = X_check.max()
    mean_val = X_check.mean()

    print(f"{dataset_name} — max: {max_val:.2f}, mean: {mean_val:.4f}")

    # Raw counts have integer values and high max
    # Log-normalised data has max typically < 10
    if max_val > 20:
        print(f"  Detected as raw counts — normalising...")
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        new_max = adata.X[0:100].max() if not issparse(adata.X) else adata.X[0:100].toarray().max()
        print(f"  Normalised — new max: {new_max:.2f}")
    else:
        print(f"  Detected as already normalised — skipping")

    gc.collect()
    return adata

adata1 = check_and_normalise(adata1, "GSE114725")
adata2 = check_and_normalise(adata2, "GSE176078")

In [ ]:
# Save raw counts before HVG selection and scaling
adata1.write(PROCESSED_DIR / "GSE114725_phase1_v2_rawcounts.h5ad", compression="gzip")
adata2.write(PROCESSED_DIR / "GSE176078_phase1_v2_rawcounts.h5ad", compression="gzip")
print("Saved (NOTE: this file is log-normalised, NOT integer counts — see project notes. Do not use for PyDESeq2.)")

In [ ]:
# ----------------------------
# Cell 10 — Store raw and select HVGs for PCA
# ----------------------------
# Store full gene normalised matrix before HVG selection
# adata.raw used for CellTypist and marker validation
adata1.raw = adata1
adata2.raw = adata2
print("Raw counts stored in adata.raw")
print(f"GSE114725 raw shape: {adata1.raw.X.shape}")
print(f"GSE176078 raw shape: {adata2.raw.X.shape}")

# Select HVGs for scaling/PCA/Harmony only
sc.pp.highly_variable_genes(adata1, n_top_genes=2000, flavor="seurat")
sc.pp.highly_variable_genes(adata2, n_top_genes=2000, flavor="seurat")

adata1 = adata1[:, adata1.var.highly_variable].copy()
adata2 = adata2[:, adata2.var.highly_variable].copy()

print(f"\nGSE114725 HVGs: {adata1.n_vars}")
print(f"GSE176078 HVGs: {adata2.n_vars}")

gc.collect()

In [ ]:
# ----------------------------
# Cell 11 — Scale, PCA, Harmony
# FIX: to_dense_chunks simplified — no dask branch needed, adata2 is
# only 2000 genes wide at this point so a direct .toarray() is fine.
# ----------------------------

# Scale
sc.pp.scale(adata1, max_value=10)
sc.pp.scale(adata2, max_value=10)
print("Scaling complete")
gc.collect()

# PCA on both datasets — standard (arpack), seeded, with a memory-safe
# fallback to IncrementalPCA for adata2 if it doesn't fit in memory.
# adata1 has never needed the fallback except during one severely
# memory-degraded session; adata2 (91k cells) is closer to the edge and
# may need it depending on available memory on the day.
sc.tl.pca(adata1, svd_solver="arpack", random_state=0)
print("adata1 PCA complete (standard)")
gc.collect()

try:
    sc.tl.pca(adata2, svd_solver="arpack", random_state=0)
    print("adata2 PCA complete (standard)")
except MemoryError:
    print("adata2 standard PCA failed — falling back to IncrementalPCA")
    from sklearn.decomposition import IncrementalPCA

    def incremental_pca_fit_transform(adata, n_comps=50, batch_size=2000, label=""):
        ipca = IncrementalPCA(n_components=n_comps, batch_size=batch_size)
        X = adata.X
        n_cells = adata.n_obs
        for start in range(0, n_cells, batch_size):
            end = min(start + batch_size, n_cells)
            chunk = X[start:end]
            if not isinstance(chunk, np.ndarray):
                chunk = np.asarray(chunk, dtype=np.float32)
            ipca.partial_fit(chunk)
        X_pca = np.zeros((n_cells, n_comps), dtype=np.float32)
        for start in range(0, n_cells, batch_size):
            end = min(start + batch_size, n_cells)
            chunk = X[start:end]
            if not isinstance(chunk, np.ndarray):
                chunk = np.asarray(chunk, dtype=np.float32)
            X_pca[start:end] = ipca.transform(chunk)
        adata.obsm["X_pca"] = X_pca
        adata.varm["PCs"] = ipca.components_.T.astype(np.float32)
        adata.uns["pca"] = {
            "variance": ipca.explained_variance_,
            "variance_ratio": ipca.explained_variance_ratio_,
        }
        return adata

    adata2 = incremental_pca_fit_transform(adata2, label="adata2")
    print("adata2 PCA complete (IncrementalPCA fallback)")

gc.collect()

# Harmony
# FIX (reproducibility): random_state added. Harmony initialises using
# sklearn.KMeans internally — without a fixed seed this is genuinely
# random each run, producing a DIFFERENT corrected embedding every time,
# which cascades into different Leiden clusters downstream and silently
# breaks the hardcoded cluster_labels_1/cluster_labels_2 dictionaries in
# notebook 04. Caught when GSE114725 went from 6 clusters at res 0.6 to
# 13 clusters at res 0.6 across two otherwise-identical re-runs.
sce.pp.harmony_integrate(adata1, key="patient", basis="X_pca", random_state=0)
sce.pp.harmony_integrate(adata2, key="orig.ident", basis="X_pca", random_state=0)
print("Harmony complete")
print(adata1.obsm.keys())
print(adata2.obsm.keys())
gc.collect()

In [ ]:
# ----------------------------
# Cell 12 — UMAP before and after Harmony
# ----------------------------
# Before Harmony
for adata, name, batch_key in [
    (adata1, "GSE114725", "patient"),
    (adata2, "GSE176078", "orig.ident")
]:
    sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=15, n_pcs=30)
    sc.tl.umap(adata, random_state=42)
    sc.pl.umap(
        adata,
        color=[batch_key],
        title=f"{name} before Harmony",
        show=False
    )
    plt.savefig(
        FIGURE_DIR / f"{name}_before_harmony.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()
    gc.collect()

# After Harmony
for adata, name, batch_key in [
    (adata1, "GSE114725", "patient"),
    (adata2, "GSE176078", "orig.ident")
]:
    sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30)
    sc.tl.umap(adata, random_state=42)
    sc.pl.umap(
        adata,
        color=[batch_key],
        title=f"{name} after Harmony",
        show=False
    )
    plt.savefig(
        FIGURE_DIR / f"{name}_after_harmony.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close()
    gc.collect()

print("UMAP plots saved")

In [ ]:
# ----------------------------
# Cell 13 — Save final objects
# ----------------------------
adata1.write(
    PROCESSED_DIR / "GSE114725_phase1_v2.h5ad",
    compression="gzip"
)
print("GSE114725 saved")
gc.collect()

adata2.write(
    PROCESSED_DIR / "GSE176078_phase1_v2.h5ad",
    compression="gzip"
)
print("GSE176078 saved")
gc.collect()

print("\nFinal objects:")
print(adata1)
print(adata2)

In [ ]:
print("GSE114725 raw:", adata1.raw)
print("GSE114725 raw shape:", adata1.raw.X.shape)
print("GSE176078 raw:", adata2.raw)
print("GSE176078 raw shape:", adata2.raw.X.shape)

In [ ]:
# ----------------------------
# Cell 14 — Pytest tests
# ----------------------------
import numpy as np
import pandas as pd
from anndata import AnnData

all_prefixes_test = (
    "IGHV", "IGLV", "IGKV", "TRAV", "TRBV", "TRGV", "TRDV",
    "IGHD", "IGHJ", "IGLJ", "IGKJ",
    "TRAC", "TRBC", "TRGC", "TRDC",
    "IGHA", "IGHE", "IGHG", "IGHM", "IGLC", "IGKC"
)

def test_vdj_constant_filter():
    """V(D)J and constant region genes should be removed"""
    X = np.random.poisson(1, (5, 6)).astype(np.float32)
    var = pd.DataFrame(
        index=["CD3D", "IGHV1-2", "MS4A1", "TRAC", "IGKC", "FOXP3"]
    )
    adata = AnnData(X, var=var)
    mask = ~adata.var_names.str.startswith(all_prefixes_test)
    filtered = adata[:, mask]
    assert "IGHV1-2" not in filtered.var_names
    assert "TRAC" not in filtered.var_names
    assert "IGKC" not in filtered.var_names
    assert "CD3D" in filtered.var_names
    assert "FOXP3" in filtered.var_names
    assert filtered.n_vars == 3
    print("test_vdj_constant_filter passed")

def test_mt_gene_removal():
    """Mitochondrial genes should be removed from var"""
    X = np.random.poisson(1, (5, 4)).astype(np.float32)
    var = pd.DataFrame(index=["CD3D", "MT-CO1", "MS4A1", "MT-ND1"])
    adata = AnnData(X, var=var)
    mask = ~adata.var_names.str.startswith("MT-")
    filtered = adata[:, mask]
    assert "MT-CO1" not in filtered.var_names
    assert "MT-ND1" not in filtered.var_names
    assert "CD3D" in filtered.var_names
    assert filtered.n_vars == 2
    print("test_mt_gene_removal passed")

def test_gene_count_filter():
    """Cells with too few genes should be removed"""
    X = np.random.poisson(1, (4, 10)).astype(np.float32)
    obs = pd.DataFrame({
        "n_genes_by_counts": [50, 150, 300, 500]
    })
    adata = AnnData(X, obs=obs)
    filtered = adata[adata.obs["n_genes_by_counts"] > 200]
    assert filtered.n_obs == 2
    print("test_gene_count_filter passed")

def test_raw_stored():
    """adata.raw should be set and contain full gene matrix"""
    X = np.abs(np.random.randn(5, 100)).astype(np.float32)
    adata = AnnData(X)
    adata.raw = adata
    assert adata.raw is not None
    assert adata.raw.X.shape[1] == 100
    print("test_raw_stored passed")

def test_normalisation_check():
    """Normalisation should not run on already normalised data"""
    # Simulate already normalised data (max < 100)
    X = np.random.uniform(0, 9, (5, 10)).astype(np.float32)
    max_val = float(X.max())
    assert max_val < 100, "Data appears already normalised"
    print("test_normalisation_check passed")

def test_no_dask_dependency():
    """Confirm adata.X is plain scipy sparse, not a dask array (regression test for the MemoryError fix)"""
    from scipy.sparse import csr_matrix
    X = csr_matrix(np.random.poisson(1, (5, 5)).astype(np.float32))
    adata = AnnData(X)
    assert not hasattr(adata.X, "compute"), "adata.X should not be a dask array"
    print("test_no_dask_dependency passed")

# Run all tests
test_vdj_constant_filter()
test_mt_gene_removal()
test_gene_count_filter()
test_raw_stored()
test_normalisation_check()
test_no_dask_dependency()
print("\nAll tests passed")